# DIMER Language-Model Fine-Tuning
**Profile:** `E2E`  
**Notebook specification:** DIMER Notebook Specification `1.0`

This notebook exercises the production `language-model-finetuner` path for data handling, assistant masking, QLoRA/PEFT training, generation, artifact staging, and fresh reconstruction. A successful run does **not** establish benchmark accuracy, factual correctness, safety, fairness, calibration, robustness, or production fitness.

[Pipeline repository](https://github.com/kurtvalcorza/language-model-pipeline) · [SmolLM2-360M](https://huggingface.co/HuggingFaceTB/SmolLM2-360M-Instruct) · [Filipino SFT dataset card](https://huggingface.co/datasets/jpaulpoliquit/ph-sft-ai-authored-v1) · [PEFT](https://huggingface.co/docs/peft)

The immutable pipeline and finetuner **runtime-source revisions** are recorded separately from the moving notebook/PR head.


## Prerequisites and data handling
Use Google Colab with CUDA. **Private production source:** `language-model-finetuner` requires a `GITHUB_TOKEN` Colab Secret with **read access** to the private repository. Authentication uses an ephemeral Git header; the token is not printed, embedded in a URL, or persisted.

The default sample is tutorial/sanity data, not benchmark evidence. BYOD stays in the hosted runtime; do not upload confidential, personal, restricted, or sensitive data unless authorized.


In [ ]:
%pip -q install transformers==5.16.1 tokenizers==0.23.2 huggingface-hub==1.30.0 peft==0.20.0 accelerate==1.14.0 bitsandbytes==0.49.0 safetensors==0.8.0 datasets==4.8.5 pandas==2.3.3 PyYAML==6.0.3 Jinja2==3.1.6
%pip -q install --no-deps git+https://github.com/kurtvalcorza/language-model-pipeline.git@afaf1f032cd7e9751db1ee8eb71b542f9bd0b15f


In [ ]:
import gc, json, shutil, sys
from pathlib import Path
import pandas as pd
import torch
from datasets import load_dataset
from lmpipeline.errors import Code, DatasetError
from lmpipeline.tutorial_runtime import checkout_private_finetuner, github_token_from_runtime
from lmpipeline.tutorial_api import (
    assert_finetuner_checkout, assert_no_split_leakage, assert_tutorial_runtime,
    canonical_dataset_digest, normalize_records, resolve_tutorial_model, seed_everything,
    sha256_file, zip_directory,
)

PIPELINE_RUNTIME_REVISION = "afaf1f032cd7e9751db1ee8eb71b542f9bd0b15f"
FINETUNER_RUNTIME_REVISION = "3772f0ca4e0f7130ffd5f826ea40f43b7212339e"
FINETUNER_ROOT = Path("/content/language-model-finetuner")
_GITHUB_TOKEN = github_token_from_runtime()
checkout_private_finetuner(FINETUNER_ROOT, revision=FINETUNER_RUNTIME_REVISION, token=_GITHUB_TOKEN)
del _GITHUB_TOKEN
sys.path.insert(0, str(FINETUNER_ROOT / "src"))

from finetuner.artifacts import build_provenance, stage_artifact, verify_manifest
from finetuner.backends import attach_adapter, load_base_model, load_tokenizer, trainable_parameter_summary
from finetuner.config import TrainingConfig
from finetuner.data import NormalizedSplits, dataset_digest, load_normalized_splits, tokenize_splits
from finetuner.inference import generate_reply, load_adapter_for_inference, verify_adapter_active
from finetuner.masking import build_masked_example
from finetuner.training import train

RUNTIME = assert_tutorial_runtime()
assert_finetuner_checkout(FINETUNER_ROOT, FINETUNER_RUNTIME_REVISION)
if not torch.cuda.is_available():
    raise RuntimeError("QLoRA requires CUDA; select a Colab GPU runtime.")
print(json.dumps({"pipelineRuntimeRevision": PIPELINE_RUNTIME_REVISION, "finetunerRuntimeRevision": FINETUNER_RUNTIME_REVISION, "runtime": RUNTIME}, indent=2))


## 1. Resolve the canonical model and seed stochastic operations
The canonical registry is the source of model identity and immutable revision. Seeding occurs before model/adapter construction; GPU reductions and quantized kernels may still vary across hardware, so bitwise determinism is not claimed.


In [ ]:
BASE_MODEL_KEY = "smollm2-360m" # @param {type:"string"}
MAX_SEQUENCE_LENGTH = 512 # @param {type:"integer"}
EPOCHS = 1 # @param {type:"integer"}
LEARNING_RATE = 0.0002 # @param {type:"number"}
LORA_RANK = 8 # @param {type:"integer"}
LORA_ALPHA = 16 # @param {type:"integer"}
SEED = 42 # @param {type:"integer"}

DETERMINISM = seed_everything(SEED)
ENTRY = resolve_tutorial_model(BASE_MODEL_KEY, method="qlora", max_sequence_length=MAX_SEQUENCE_LENGTH)
TOKENIZER = load_tokenizer(ENTRY)
print(json.dumps(DETERMINISM, indent=2))
print({"modelKey": ENTRY.key, "modelId": ENTRY.model_id, "revision": ENTRY.revision, "license": ENTRY.license})


## 2. Load pinned sample or BYOD through the production data path
The default public sample is pinned by dataset revision. BYOD accepts the production JSONL/ZIP forms. Split derivation is deterministic when validation is absent, leakage is rejected, and over-length examples fail rather than being silently truncated.


In [ ]:
DATA_SOURCE = "Sample: Filipino SFT" # @param ["Sample: Filipino SFT","Bring Your Own Dataset"]
SAMPLE_LIMIT = 120 # @param {type:"integer"}
WORK_DIR = Path("/content/language-model-tutorial")
shutil.rmtree(WORK_DIR, ignore_errors=True)
WORK_DIR.mkdir(parents=True)

if DATA_SOURCE == "Sample: Filipino SFT":
    dataset_id = "jpaulpoliquit/ph-sft-ai-authored-v1"
    dataset_revision = "8333699c6cc7296cc69cefc09def010851ded919"
    rows = [dict(row) for row in load_dataset(dataset_id, revision=dataset_revision, split="train")]
    normalized = sorted(normalize_records(rows), key=lambda item: item.fingerprint())
    selected = []
    for item in normalized:
        if len(selected) >= SAMPLE_LIMIT:
            break
        try:
            build_masked_example(TOKENIZER, list(item.messages), line_number=item.line_number, max_sequence_length=MAX_SEQUENCE_LENGTH)
        except DatasetError as exc:
            if exc.code == Code.DATASET_SEQUENCE_TOO_LONG:
                continue
            raise
        selected.append(item)
    NORMALIZED = NormalizedSplits(splits={"train": selected}, source=f"{dataset_id}@{dataset_revision}", archive=None)
    DATASET_DIGEST = canonical_dataset_digest(NORMALIZED.splits)
    DATASET_PROVENANCE = {"source": dataset_id, "revision": dataset_revision, "license": "apache-2.0", "usage": "tutorial-sanity-not-benchmark"}
else:
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No BYOD files uploaded")
    upload_root = WORK_DIR / "upload"
    upload_root.mkdir()
    for name, payload in uploaded.items():
        (upload_root / Path(name).name).write_bytes(payload)
    NORMALIZED = load_normalized_splits(upload_root, workdir=WORK_DIR / "resolved")
    DATASET_DIGEST = dataset_digest(upload_root, WORK_DIR / "digest-resolved")
    DATASET_PROVENANCE = {"source": "BYOD", "transport": NORMALIZED.source, "archive": NORMALIZED.archive, "usage": "user-provided"}

assert_no_split_leakage(NORMALIZED.splits)
print({name: len(items) for name, items in NORMALIZED.splits.items()}, "datasetDigest", DATASET_DIGEST)


## 3. Tokenize and mask with production code
Assistant response tokens receive causal cross-entropy loss. User, system, chat-template, and padding tokens are masked with `-100`. The same tokenizer/chat template is serialized with the adapter, preserving training/inference formatting.


In [ ]:
SPLITS = tokenize_splits(
    NORMALIZED, tokenizer=TOKENIZER, max_sequence_length=MAX_SEQUENCE_LENGTH,
    validation_fraction=0.20, seed=SEED,
)
print("effective splits", SPLITS.counts(), "validation derived", SPLITS.validation_was_derived)
print("supervised tokens", {
    "train": sum(x.supervised_token_count for x in SPLITS.train),
    "validation": sum(x.supervised_token_count for x in SPLITS.validation),
    "test": sum(x.supervised_token_count for x in SPLITS.test),
})


## 4. Record a deterministic pre-adaptation baseline
`finetuner.backends.load_base_model` owns production QLoRA loading and `finetuner.inference.generate_reply` owns generation. Greedy decoding (`do_sample=False`) is used for reproducible baseline/adapted comparisons; application sampling should record temperature/top-p/top-k explicitly.


In [ ]:
LOADED = load_base_model(ENTRY, method="qlora", device="cuda", tokenizer=TOKENIZER)
BASE_EMBEDDING_SIZE = LOADED.model.get_input_embeddings().num_embeddings
PROBE_PROMPTS = [
    "Ipaliwanag sa simpleng Filipino kung ano ang machine learning.",
    "Magbigay ng tatlong paraan para mabawasan ang basura sa opisina.",
]
BASELINE_OUTPUTS = [generate_reply(LOADED.model, TOKENIZER, p, max_new_tokens=96, decoding={"do_sample": False}) for p in PROBE_PROMPTS]
display(pd.DataFrame({"prompt": PROBE_PROMPTS, "base": BASELINE_OUTPUTS}))


## 5. Train through the production finetuner

The production scheduler, warmup, gradient clipping, weight decay, validation, early stopping, and optional best-adapter restoration controls are used.

### How to read the principal metrics
- **`trainLoss`**: mean causal cross-entropy per supervised **assistant token** on training data. Lower means better fit to those tokens; it is not accuracy.
- **`validationLoss`**: the same token-weighted loss on the held-out validation split without gradient updates. It helps detect overfitting/choose an epoch, but a small tutorial holdout is not a task benchmark.
- **`testLoss`**: the same measure on an explicitly supplied independent test split after selection/restoration; it is `null` on the default sample because no independent test split is supplied.
- **`trainPerplexity` / `validationPerplexity` / `testPerplexity`**: `exp(loss)` when finite; they summarize token-level uncertainty, not factual correctness or end-user task quality.
- **`examplesPerSecond` / `wallSeconds`**: throughput/runtime for this hardware, sequence-length distribution, and software stack.
- **`peakGpuMemoryBytes`**: observed peak allocated GPU memory, useful for capacity diagnostics, not model quality.
- **`epochsCompleted`, `optimizerSteps`, and `trainingControls`**: execution/control provenance describing how far training ran and which selection/scheduler controls actually applied.

These metrics are **optimization evidence**. Loss/perplexity shown here are **sample/tutorial optimization metrics**, not sufficient task-quality evidence.


In [ ]:
MODEL = attach_adapter(LOADED, rank=LORA_RANK, alpha=LORA_ALPHA, dropout=0.05)
print("trainable parameters", trainable_parameter_summary(MODEL))
TRAINING_CONFIG = TrainingConfig(
    method="qlora", epochs=EPOCHS, learning_rate=LEARNING_RATE,
    lora_rank=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=0.05,
    per_device_batch_size=1, gradient_accumulation_steps=2, seed=SEED,
    max_sequence_length=MAX_SEQUENCE_LENGTH, validation_split=0.20,
    weight_decay=0.01, lr_scheduler_type="constant", warmup_ratio=0.0,
    early_stopping_patience=0, early_stopping_min_delta=0.0,
    restore_best_adapter=False,
)
METRICS = train(MODEL, SPLITS, config=TRAINING_CONFIG, pad_token_id=TOKENIZER.pad_token_id, device="cuda").to_dict()
ADAPTED_OUTPUTS = [generate_reply(MODEL, TOKENIZER, p, max_new_tokens=96, decoding={"do_sample": False}) for p in PROBE_PROMPTS]
print(json.dumps(METRICS, indent=2))
display(pd.DataFrame({"prompt": PROBE_PROMPTS, "base": BASELINE_OUTPUTS, "adapted": ADAPTED_OUTPUTS}))


## 6. Run real new-input inference and export machine-readable outputs
Edit `CUSTOM_PROMPT`; it is separate from the training/validation sample. JSONL and CSV preserve input identity, prompt, outputs, exact model identity, and decoding. These files may contain sensitive prompt/output text when BYOD or sensitive prompts are used.


In [ ]:
CUSTOM_PROMPT = "Sumulat ng dalawang pangungusap tungkol sa responsableng paggamit ng AI." # @param {type:"string"}
NEW_PROMPTS = PROBE_PROMPTS + [CUSTOM_PROMPT]
RESULT_ROWS = []
for index, prompt in enumerate(NEW_PROMPTS, 1):
    baseline = BASELINE_OUTPUTS[index - 1] if index <= len(BASELINE_OUTPUTS) else None
    adapted = generate_reply(MODEL, TOKENIZER, prompt, max_new_tokens=96, decoding={"do_sample": False})
    RESULT_ROWS.append({"inputId": f"prompt-{index}", "prompt": prompt, "base": baseline, "adapted": adapted, "modelId": ENTRY.model_id, "modelRevision": ENTRY.revision, "decoding": {"doSample": False, "maxNewTokens": 96}})
OUTPUT_JSONL = WORK_DIR / "tutorial_predictions.jsonl"
OUTPUT_JSONL.write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in RESULT_ROWS) + "\n", encoding="utf-8")
OUTPUT_CSV = WORK_DIR / "tutorial_predictions.csv"
pd.DataFrame(RESULT_ROWS).to_csv(OUTPUT_CSV, index=False)
METRICS_JSON = WORK_DIR / "tutorial_metrics.json"
METRICS_JSON.write_text(json.dumps(METRICS, indent=2), encoding="utf-8")
display(pd.DataFrame(RESULT_ROWS)[["inputId", "prompt", "base", "adapted"]])
print("wrote", OUTPUT_JSONL, OUTPUT_CSV, METRICS_JSON)


## 7. Export and fresh-reload the deployable artifact
The deliverable is a **PEFT adapter, not a complete model**. It cannot run by itself: reconstruction requires the exact base model and immutable revision recorded in `provenance.json`. The bundle contains adapter weights/config, tokenizer/chat-template assets, metrics, provenance, a model card, and `artifact-manifest.json`; it deliberately does not copy base weights or raw training rows.

The manifest identifies `format = peft_adapter`, `formatVersion = 1` and hashes the load-bearing files. Weights use safetensors/JSON rather than pickle state. Although raw training rows are not exported, the adapter is learned from the source dataset and remains subject to applicable confidentiality, retention, licensing, and disclosure obligations.

Fresh reconstruction verifies non-zero LoRA B matrices and a non-zero adapter-on/off logit delta on the same reloaded model. That proves the serialized adapter is active without claiming output equivalence.


In [ ]:
JOB_DICT = {"training": TRAINING_CONFIG.to_dict(), "tutorial": {"profile": "E2E", "notebookSpecVersion": "1.0"}}
PROVENANCE = build_provenance(
    entry=ENTRY, job_dict=JOB_DICT, dataset_digest=DATASET_DIGEST,
    loaded_dtype=LOADED.torch_dtype, quantized=LOADED.quantized,
    target_modules=LOADED.target_modules, dimer_base_model=None,
)
PROVENANCE["dataset"] = DATASET_PROVENANCE
PROVENANCE["runtimeRevisions"] = {"pipeline": PIPELINE_RUNTIME_REVISION, "finetuner": FINETUNER_RUNTIME_REVISION}
PROVENANCE["determinism"] = DETERMINISM
required = {"torch", "transformers", "tokenizers", "peft", "bitsandbytes", "safetensors"}
missing = required - set(PROVENANCE["packageVersions"])
if missing:
    raise RuntimeError(f"Producer provenance missing package versions: {sorted(missing)}")

STAGE = stage_artifact(
    MODEL, TOKENIZER, output_dir=WORK_DIR / "dimer-lm-adapter",
    entry=ENTRY, provenance=PROVENANCE, metrics=METRICS,
    base_embedding_size=BASE_EMBEDDING_SIZE,
)
verify_manifest(STAGE.path)
MANIFEST = json.loads((STAGE.path / "artifact-manifest.json").read_text(encoding="utf-8"))
if (MANIFEST.get("format"), MANIFEST.get("formatVersion")) != ("peft_adapter", 1):
    raise RuntimeError("Unexpected artifact manifest format/version")
ARTIFACT_ZIP = zip_directory(STAGE.path, WORK_DIR / "dimer-language-model-adapter.zip")
ARTIFACT_SHA256 = sha256_file(ARTIFACT_ZIP)

del MODEL, LOADED
gc.collect()
torch.cuda.empty_cache()
RELOADED_MODEL, RELOADED_TOKENIZER = load_adapter_for_inference(
    STAGE.path, entry=ENTRY, device="cuda", quantized=True,
)
ADAPTER_ACTIVITY = verify_adapter_active(RELOADED_MODEL, RELOADED_TOKENIZER, prompt=CUSTOM_PROMPT)
RELOADED_ANSWER = generate_reply(RELOADED_MODEL, RELOADED_TOKENIZER, CUSTOM_PROMPT, decoding={"do_sample": False})
if not RELOADED_ANSWER:
    raise RuntimeError("Fresh reconstruction produced an empty answer")
print(json.dumps({"artifactFormat": MANIFEST["format"], "artifactFormatVersion": MANIFEST["formatVersion"], "artifactSha256": ARTIFACT_SHA256, "adapterActivity": ADAPTER_ACTIVITY, "freshAnswer": RELOADED_ANSWER}, indent=2, ensure_ascii=False))

from google.colab import files
files.download(str(ARTIFACT_ZIP))


## Interpretation, troubleshooting, resources, and next experiments
A successful E2E run proves this tutorial path can normalize/tokenize data, train with production code, generate new-input predictions, serialize a versioned PEFT adapter, reconstruct it against its exact base revision, and demonstrate active adapter deltas. It does **not** prove task quality or production fitness.

Common failures: fix `GITHUB_TOKEN` access rather than embedding credentials; select a CUDA runtime when unavailable; shorten over-length records rather than silently truncating; remove cross-split duplicates; reduce sequence/batch demands or use suitable hardware for OOM; reject manifest/reload failures rather than publishing an unverifiable artifact.

Next experiments: enable early stopping/best-adapter restoration over multiple epochs; use authorized BYOD plus an independent test split and task-specific rubric/human review; compare another **user-facing** registry model under fixed data/split/decoding; compare deterministic verification with explicitly recorded stochastic decoding.

Source links: [pipeline](https://github.com/kurtvalcorza/language-model-pipeline), [private finetuner](https://github.com/kurtvalcorza/language-model-finetuner), [SmolLM2-360M](https://huggingface.co/HuggingFaceTB/SmolLM2-360M-Instruct), [Filipino SFT dataset](https://huggingface.co/datasets/jpaulpoliquit/ph-sft-ai-authored-v1), [PEFT](https://huggingface.co/docs/peft), [release verification](RELEASE_VERIFICATION.md).

Release-grade status additionally requires the clean-runtime record: execute this exact candidate head in a clean supported Colab GPU runtime, preserve its ZIP/digest, then execute the companion `ARTIFACT-INFERENCE` notebook in a **separate clean runtime** with that external ZIP. Static CI is not REL1/REL5 execution evidence.
